In [46]:
import pandas as pd
import numpy as np

In [47]:
# Custom functions
from core.demand_calculations import *

# Energy

## Capacity technology data

In [48]:
cap_tech_df = pd.read_excel(r'data/Demand.xlsx', sheet_name='ELEC_CAP_TECH')
mi_df = pd.read_excel(r'data/Demand.xlsx', sheet_name='MI')

PermissionError: [Errno 13] Permission denied: 'data/Demand.xlsx'

In [35]:
cap_tech_can_df = cap_tech_df[cap_tech_df['Region'] == 'Canada']
cap_tech_can_df

,Scenario,Region,Variable,Year,Value,Unit
775,Canada Net-zero,Canada,Battery Storage,2020,95.0,MW
776,Canada Net-zero,Canada,Battery Storage,2021,95.0,MW
777,Canada Net-zero,Canada,Battery Storage,2022,140.0,MW
778,Canada Net-zero,Canada,Battery Storage,2023,140.0,MW
779,Canada Net-zero,Canada,Battery Storage,2024,140.0,MW
...,...,...,...,...,...,...
9481,Global Net-zero,Canada,Uranium SMR,2046,22671.0,MW
9482,Global Net-zero,Canada,Uranium SMR,2047,22996.0,MW
9483,Global Net-zero,Canada,Uranium SMR,2048,23311.0,MW
9484,Global Net-zero,Canada,Uranium SMR,2049,23627.0,MW


In [23]:
# Create a df with the unique variable names
variables_df = pd.DataFrame(cap_tech_can_df['Variable'].unique(), columns=['Variable'])
variables_df

,Variable
0,Battery Storage
1,Bioenergy
2,Bioenergy with CCUS
3,Coal and Coke
4,Coal with CCUS
5,Hydro
6,Hydrogen
7,Natural Gas
8,Natural Gas with CCUS
9,Offshore Wind


## Define mapping dictionnary between CER technologies and MI sub-technologies

In [25]:
mapping_dict = {
    "Solar (Distributed)": [("Sol_C-si", 1.0)],
    "Solar (Utility scale)": [("Sol_C-si", 0.8), ("Sol_Thin_Film", 0.2)],
    "Onshore Wind": [("Wind_Onshore", 1.0)],
    "Offshore Wind": [
        ("Wind_DD-PMSG_Offshore", 0.6),
        ("Wind_GB-PMSG_Offshore", 0.2),
        ("Wind_GB-DFIG_SCIG_Offshore", 0.2),
    ],
    "Hydro": [("Hydro", 1.0)],
    #"Battery Storage": [("Battery_Li-ion", 1.0)],
    "Bioenergy": [("Bioenergy", 1.0)],
    "Bioenergy with CCUS": [("Bioenergy_CCUS", 1.0)],
    "Coal and Coke": [("Coal", 1.0)],
    "Coal with CCUS": [("Coal_CCUS", 1.0)],
    "Hydrogen": [("Hydrogen_Electrolysis", 1.0)],
    "Natural Gas": [("Gas", 1.0)],
    "Natural Gas with CCUS": [("Gas_CCUS", 1.0)],
    "Oil": [("Oil", 1.0)],
    "Uranium": [("Nuclear_Conventional", 1.0)],
    "Uranium SMR": [("Nuclear_SMR", 1.0)],
}

## Disaggregate capacity data based on the mapping dictionary

In [29]:
def disaggregate_capacity(df, mapping_dict,
                                           tech_col="Variable",
                                           cap_col="Value",
                                           new_col="Associated_MI"):
    """
    Expand capacity data by linking each CER technology to one or more MI sub-technologies.

    Keeps the original Variable column unchanged and adds a new column for the associated MI.
    The capacity is split across sub-technologies according to the specified shares.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe (e.g., ELEC_CAP_TECH).
    mapping_dict : dict
        { main_tech : [(mi_tech, share), ...] } mapping.
    tech_col : str
        Column containing CER technology names.
    cap_col : str
        Column with capacity values (GW).
    new_col : str
        New column name to store the associated MI bucket.

    Returns
    -------
    pd.DataFrame
        Disaggregated dataframe with added Associated_MI and scaled capacity values.
    """
    rows = []
    for _, row in df.iterrows():
        tech = row[tech_col]
        if tech in mapping_dict:
            for mi_tech, share in mapping_dict[tech]:
                new_row = row.copy()
                new_row[new_col] = mi_tech
                new_row[cap_col] = row[cap_col] * share
                rows.append(new_row)
        else:
            # Keep row unchanged but mark as unmapped
            new_row = row.copy()
            new_row[new_col] = tech  # or could be None
            rows.append(new_row)
    expanded_df = pd.DataFrame(rows)
    return expanded_df


In [36]:
disagg_df = disaggregate_capacity(cap_tech_can_df, mapping_dict, tech_col="Variable", cap_col="Value")

In [37]:
disagg_df

,Scenario,Region,Variable,Year,Value,Unit,Associated_MI
775,Canada Net-zero,Canada,Battery Storage,2020,95.0,MW,Battery_Li-ion
776,Canada Net-zero,Canada,Battery Storage,2021,95.0,MW,Battery_Li-ion
777,Canada Net-zero,Canada,Battery Storage,2022,140.0,MW,Battery_Li-ion
778,Canada Net-zero,Canada,Battery Storage,2023,140.0,MW,Battery_Li-ion
779,Canada Net-zero,Canada,Battery Storage,2024,140.0,MW,Battery_Li-ion
...,...,...,...,...,...,...,...
9481,Global Net-zero,Canada,Uranium SMR,2046,22671.0,MW,Nuclear_SMR
9482,Global Net-zero,Canada,Uranium SMR,2047,22996.0,MW,Nuclear_SMR
9483,Global Net-zero,Canada,Uranium SMR,2048,23311.0,MW,Nuclear_SMR
9484,Global Net-zero,Canada,Uranium SMR,2049,23627.0,MW,Nuclear_SMR


In [38]:
# Put the value in GW and
disagg_df['Value'] = disagg_df['Value'] / 1000
disagg_df['Unit'] = 'GW'

In [39]:
disagg_df

,Scenario,Region,Variable,Year,Value,Unit,Associated_MI
775,Canada Net-zero,Canada,Battery Storage,2020,0.095,GW,Battery_Li-ion
776,Canada Net-zero,Canada,Battery Storage,2021,0.095,GW,Battery_Li-ion
777,Canada Net-zero,Canada,Battery Storage,2022,0.140,GW,Battery_Li-ion
778,Canada Net-zero,Canada,Battery Storage,2023,0.140,GW,Battery_Li-ion
779,Canada Net-zero,Canada,Battery Storage,2024,0.140,GW,Battery_Li-ion
...,...,...,...,...,...,...,...
9481,Global Net-zero,Canada,Uranium SMR,2046,22.671,GW,Nuclear_SMR
9482,Global Net-zero,Canada,Uranium SMR,2047,22.996,GW,Nuclear_SMR
9483,Global Net-zero,Canada,Uranium SMR,2048,23.311,GW,Nuclear_SMR
9484,Global Net-zero,Canada,Uranium SMR,2049,23.627,GW,Nuclear_SMR


In [41]:
mi_df

,Technology,Sub-technology,Metal,Metal intensity,Unit,Reference,Comment
0,Biomass,Aggregated,Aluminum,3900,t/GW,"(Ashby, 2013)",NaN
1,Biomass,Aggregated,Aluminum,1300,t/GW,"(Sullivan et al., 2010)",NaN
2,Biomass,Aggregated,Aluminum,271.2,t/GW,"(Van Oorschot et al., 2022), (Bauer 2007, Moss...",NaN
3,Biomass,Wood energy,Aluminum,184,t/GW,Open LCA Analysis 'Wood energy',NaN
4,Biomass,Aggregated,Chromium,2,t/GW,"(Ashby, 2013)",NaN
...,...,...,...,...,...,...,...
708,Wind,GB-PMSG,Zinc,5500,t/GW,"(European Commission, 2020)",Hybride
709,Wind,Offshore,Zinc,5500,t/GW,"(IEA, 2022)",NaN
710,Wind,Offshore,Zinc,5450,t/GW,(Watari et al. 2019),NaN
711,Wind,Onshore,Zinc,5500,t/GW,"(IEA, 2022)",NaN


In [40]:
def compute_metal_demand(disagg_df, mi_df,
                         mi_col="Associated_MI",
                         cap_col="Value",
                         metal_col="Metal",
                         intensity_col="Metal intensity",
                         unit_col="Unit"):
    """
    Merge disaggregated electricity capacity data with metal intensity data
    and compute total metal requirements (e.g., tonnes per year, province, scenario).

    Parameters
    ----------
    disagg_df : pd.DataFrame
        Disaggregated capacity dataframe with columns like:
        ['Scenario', 'Region', 'Variable', 'Year', 'Associated_MI', 'Value', 'Unit'].
        'Value' must be in GW.
    mi_df : pd.DataFrame
        Metal intensity dataframe with columns like:
        ['Technology', 'Metal', 'Intensity_t_per_GW'].
    mi_col : str
        Column in disagg_df identifying the associated MI bucket.
    cap_col : str
        Capacity column (in GW).
    metal_col : str
        Column in mi_df identifying the metal name.
    intensity_col : str
        Column in mi_df containing metal intensity (t/GW).
    unit_col : str
        Column name in disagg_df indicating the energy unit (default 'Unit').

    Returns
    -------
    pd.DataFrame
        DataFrame with added columns:
        ['Metal', 'Metal_Unit', 'Metal_Value_t'] plus all original context columns.
    """
    # --- Standardize column names for merging ---
    mi_df = mi_df.rename(columns={
        "Technology": mi_col,  # match Associated_MI in disagg_df
        "Intensity_t_per_GW": intensity_col,
    })

    # --- Merge disaggregated data with metal intensity table ---
    merged = disagg_df.merge(mi_df, on=mi_col, how="left")

    # --- Compute total metal demand (t) ---
    merged["Metal_Value_t"] = merged[cap_col] * merged[intensity_col]
    merged["Metal_Unit"] = "t"

    # --- Optional clean-up: drop rows with no intensity data ---
    merged = merged.dropna(subset=["Metal_Value_t"])

    return merged


ModuleNotFoundError: No module named 'caas_jupyter_tools'

In [ ]:

# Example of harmonizing MI sheet column names if needed
# (Your MI sheet may use slightly different names)
mi_df = mi_df.rename(columns={
    "MI_Technology": "Technology",
    "Metal intensity (t/GW)": "Intensity_t_per_GW"
})

# Run the calculation
metal_demand_df = compute_metal_demand(disagg_df, mi_df)


# EVs

## Import data

In [63]:
# EV sales data in Canada from IEA
ev_df = pd.read_excel(r'data/Demand.xlsx', sheet_name='EVs_CAN')

In [66]:
ev_df = ev_df[
    (ev_df['Parameter'] == 'EV sales') &
    (ev_df['Year'] == 2024)
]

In [67]:
ev_df

,Country,Parameter,Mode,Powertrain,Year,Unit,Value
3,Canada,EV sales,2 and 3 wheelers,BEV,2024,Vehicles,170.0
22,Canada,EV sales,Buses,BEV,2024,Vehicles,210.0
29,Canada,EV sales,Buses,FCEV,2024,Vehicles,13.0
48,Canada,EV sales,Cars,BEV,2024,Vehicles,190000.0
51,Canada,EV sales,Cars,PHEV,2024,Vehicles,62000.0
66,Canada,EV sales,Cars,FCEV,2024,Vehicles,20.0
101,Canada,EV sales,Trucks,BEV,2024,Vehicles,2000.0
105,Canada,EV sales,Trucks,FCEV,2024,Vehicles,6.0
123,Canada,EV sales,Vans,BEV,2024,Vehicles,11000.0
130,Canada,EV sales,Vans,PHEV,2024,Vehicles,6400.0


In [50]:
# End use demand for transport sector in PJ from CER scenarios
transport_demand_ef = pd.read_excel(r'data/Demand.xlsx', sheet_name='END_USE_TRANSPORT')

In [52]:
# Filter for Year 2024 and above, variable = electricity and region = Canada
transport_demand_elec_ef = transport_demand_ef[
    (transport_demand_ef['Region'] == 'Canada') &
    (transport_demand_ef['Year'] >= 2024) &
    (transport_demand_ef['Variable'] == 'Electricity')
]

In [53]:
transport_demand_elec_ef

,Scenario,Region,Variable,Year,Value,Sector,Unit
1224,Current Measures,Canada,Electricity,2024,11.5558,Transportation,PJ
1235,Current Measures,Canada,Electricity,2025,16.5362,Transportation,PJ
1246,Current Measures,Canada,Electricity,2026,22.2906,Transportation,PJ
1257,Current Measures,Canada,Electricity,2027,28.3160,Transportation,PJ
1268,Current Measures,Canada,Electricity,2028,34.8046,Transportation,PJ
...,...,...,...,...,...,...,...
18336,Global Net-zero,Canada,Electricity,2046,687.7432,Transportation,PJ
18477,Global Net-zero,Canada,Electricity,2047,709.8365,Transportation,PJ
18618,Global Net-zero,Canada,Electricity,2048,731.0948,Transportation,PJ
18759,Global Net-zero,Canada,Electricity,2049,751.4916,Transportation,PJ


## Compute future EV sales based on growth factors (based on 2024 values)

In [54]:
# Compute growth for each scenario per year, assuming 2024 value is the base
def compute_ev_growth(ev_df, base_year=2024):
    ev_growth_df = ev_df.copy()
    for scenario in ev_growth_df['Scenario'].unique():
        base_value = ev_growth_df[(ev_growth_df['Scenario'] == scenario) & (ev_growth_df['Year'] == base_year)]['Value'].values[0]
        ev_growth_df.loc[ev_growth_df['Scenario'] == scenario, 'Growth_Factor'] = ev_growth_df.loc[ev_growth_df['Scenario'] == scenario, 'Value'] / base_value
    return ev_growth_df[['Scenario', 'Year', 'Growth_Factor']]

In [55]:
ev_growth_df = compute_ev_growth(transport_demand_elec_ef)

In [56]:
ev_growth_df

,Scenario,Year,Growth_Factor
1224,Current Measures,2024,1.000000
1235,Current Measures,2025,1.430987
1246,Current Measures,2026,1.928953
1257,Current Measures,2027,2.450371
1268,Current Measures,2028,3.011873
...,...,...,...
18336,Global Net-zero,2046,40.554722
18477,Global Net-zero,2047,41.857516
18618,Global Net-zero,2048,43.111072
18759,Global Net-zero,2049,44.313827


In [68]:
ev_df

,Country,Parameter,Mode,Powertrain,Year,Unit,Value
3,Canada,EV sales,2 and 3 wheelers,BEV,2024,Vehicles,170.0
22,Canada,EV sales,Buses,BEV,2024,Vehicles,210.0
29,Canada,EV sales,Buses,FCEV,2024,Vehicles,13.0
48,Canada,EV sales,Cars,BEV,2024,Vehicles,190000.0
51,Canada,EV sales,Cars,PHEV,2024,Vehicles,62000.0
66,Canada,EV sales,Cars,FCEV,2024,Vehicles,20.0
101,Canada,EV sales,Trucks,BEV,2024,Vehicles,2000.0
105,Canada,EV sales,Trucks,FCEV,2024,Vehicles,6.0
123,Canada,EV sales,Vans,BEV,2024,Vehicles,11000.0
130,Canada,EV sales,Vans,PHEV,2024,Vehicles,6400.0


In [74]:
ev_growth_df.to_csv(r'ev_growth_df.csv', index=False)
ev_df.to_csv(r'ev_df.csv', index=False)

In [75]:
# Multiple ev_df with growth factors for each scenario and year
# Create function to apply growth factor to 2024 EV sales across scenarios and years
def project_ev_sales(ev_df, ev_growth_df):
    # Filter the base year (2024) data
    base_year_df = ev_df[ev_df["Year"] == 2024].copy()

    # Prepare to store results
    projections = []

    # Loop through each scenario
    for scenario in ev_growth_df["Scenario"].unique():
        scenario_growth = ev_growth_df[ev_growth_df["Scenario"] == scenario]

        for _, growth_row in scenario_growth.iterrows():
            year = growth_row["Year"]
            factor = growth_row["Growth_Factor"]

            # Apply the growth factor to the base EV sales data
            projected = base_year_df.copy()
            projected["Scenario"] = scenario
            projected["Year"] = year
            projected["Value"] = projected["Value"] * factor

            projections.append(projected)

    # Combine all projections into a single DataFrame
    result_df = pd.concat(projections, ignore_index=True)
    return result_df

In [76]:
# Apply the function
projected_ev_sales_df = project_ev_sales(ev_df, ev_growth_df)


In [77]:
projected_ev_sales_df

,Country,Parameter,Mode,Powertrain,Year,Unit,Value,Scenario
0,Canada,EV sales,2 and 3 wheelers,BEV,2024,Vehicles,170.000000,Current Measures
1,Canada,EV sales,Buses,BEV,2024,Vehicles,210.000000,Current Measures
2,Canada,EV sales,Buses,FCEV,2024,Vehicles,13.000000,Current Measures
3,Canada,EV sales,Cars,BEV,2024,Vehicles,190000.000000,Current Measures
4,Canada,EV sales,Cars,PHEV,2024,Vehicles,62000.000000,Current Measures
...,...,...,...,...,...,...,...,...
805,Canada,EV sales,Cars,FCEV,2050,Vehicles,909.419875,Global Net-zero
806,Canada,EV sales,Trucks,BEV,2050,Vehicles,90941.987452,Global Net-zero
807,Canada,EV sales,Trucks,FCEV,2050,Vehicles,272.825962,Global Net-zero
808,Canada,EV sales,Vans,BEV,2050,Vehicles,500180.930984,Global Net-zero
